In [ ]:
# Importación de librerías
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [ ]:
# Definir transformaciones para los datos (preprocesamiento)
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Cargar los datos
train_data = datasets.ImageFolder('data/train', transform=transform)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

val_data = datasets.ImageFolder('data/val', transform=transform)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

In [ ]:
# Usar un modelo preentrenado como ResNet
model = models.resnet50(pretrained=True)

In [ ]:
# Congelar las capas preentrenadas (opcional)
for param in model.parameters():
    param.requires_grad = False

In [ ]:
# Modificar la capa final para clasificación binaria
model.fc = nn.Linear(model.fc.in_features, 1)

In [ ]:
# Definir el optimizador y la función de pérdida
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

In [ ]:
# Entrenamiento del modelo
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs.squeeze(), labels.float())
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")

In [ ]:
# Validación del modelo
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        predicted = (outputs.squeeze() > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

In [ ]:
print(f"Accuracy: {100 * correct / total}%")